In [1]:
import pandas as pd
import seaborn as sns

In [2]:
df = sns.load_dataset('titanic')

In [3]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 80.7 KB


In [5]:
df.isnull().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [6]:
#fill missing ages with median age
df['age']=df['age'].fillna(df['age'].median())

In [7]:
#fill missing emabarked with most common port (mode)
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

In [8]:
#drop deck colum, its 77% empty, so not very useful
df=df.drop(columns = 'deck')

In [9]:
df.isnull().sum()

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    2
alive          0
alone          0
dtype: int64

In [10]:
#drop columns that are duplicates
df=df.drop(columns = ['alive', 'class', 'embark_town', 'who', 'adult_male', 'alone'])
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [11]:
#covert text to numbers and use multicollinearity while conversion 
# so if a variable has two values, it gets one column and if a column as 3 values, 
# it gets two columns
''' (multicollinearity= if there are 3 ports and you know it's "not Q and not S," then it must be C 
— so the embarked_C column is redundant. We drop one category as a reference. 
this is the exact same reason to drop a reference level to avoid the 
dummy variable trap / multicollinearity, drop_first - drop one reference category 
from each'''

df=pd.get_dummies(df, columns = ['sex', 'embarked'], drop_first = True)
df.head()

,survived,pclass,age,sibsp,parch,fare,sex_male,embarked_Q,embarked_S
0,0,3,22.0,1,0,7.2500,True,False,True
1,1,1,38.0,1,0,71.2833,False,False,False
2,1,3,26.0,0,0,7.9250,False,False,True
3,1,1,35.0,1,0,53.1000,False,False,True
4,0,3,35.0,0,0,8.0500,True,False,True


In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   survived    891 non-null    int64  
 1   pclass      891 non-null    int64  
 2   age         891 non-null    float64
 3   sibsp       891 non-null    int64  
 4   parch       891 non-null    int64  
 5   fare        891 non-null    float64
 6   sex_male    891 non-null    bool   
 7   embarked_Q  891 non-null    bool   
 8   embarked_S  891 non-null    bool   
dtypes: bool(3), float64(2), int64(4)
memory usage: 44.5 KB


In [13]:
from sklearn.model_selection import train_test_split
# features (X) = everything excepr the answe. Targer (y) = answer
X=df.drop(columns = 'survived')
y=df['survived']

#split rows - 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

print('Training passengers:', X_train.shape[0])
print('Testing passengers:', X_test.shape[0])

Training passengers: 712
Testing passengers: 179


In [14]:
#build, train and grade the model
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

#1 create model
model = LogisticRegression(max_iter=1000)

#2 train model 
model.fit(X_train, y_train)

#3 predict survival
predictions = model.predict(X_test)

#compare predictions to real values
accuracy = accuracy_score(y_test, predictions)
print('Accuracy on unseen passengers:', round(accuracy*100,1),'%')

Accuracy on unseen passengers: 81.0 %


In [15]:
y_test.value_counts(normalize=True)

survived
0    0.586592
1    0.413408
Name: proportion, dtype: float64

In [17]:
# comparision table
comparison = pd.DataFrame({
    'actual' : y_test,
    'predicted' : predictions
})

comparison['correct'] = comparison['actual'] == comparison['predicted']
comparison.head(20)

,actual,predicted,correct
709,1,0,False
439,0,0,True
840,0,0,True
720,1,1,True
39,1,1,True
290,1,1,True
300,1,1,True
333,0,0,True
208,1,1,True
136,1,1,True


In [18]:
comparison['correct'].value_counts()

correct
True     145
False     34
Name: count, dtype: int64

In [20]:
#confusion matrix
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, predictions)

array([[90, 15],
       [19, 55]])